# GC-IDM: Amortized Planning in World Models

Reproduces the main results. Trains GC-IDM, evaluates vs CEM and other solvers.

**Recommended runtime: L4 GPU** (Runtime → Change runtime type → L4). T4 works but is slower; the H5 datasets and embeddings extraction need the extra VRAM.

In [ ]:
%pip install --quiet h5py hdf5plugin scikit-learn imageio[ffmpeg] "numba<0.62.0" gdown
%pip install --quiet "stable-pretraining @ git+https://github.com/galilai-group/stable-pretraining.git"
%pip install --quiet "stable-worldmodel[train,env] @ git+https://github.com/galilai-group/stable-worldmodel.git"
%pip install --quiet mujoco dm-control

⚠️ **Restart required.** Run the cell below — the session will restart. Then continue from the next section. This is normal after installing `mujoco` / `dm-control`.

In [ ]:
import os
os.kill(os.getpid(), 9)

⬇️ **Start here after restart.** Set `ENV` again, then run all cells below.

In [ ]:
# Re-run after restart — set ENV, cd to repo, set env vars
ENV = 'tworoom'  # 'tworoom' | 'pusht' | 'cube' | 'reacher'

ENV_DATASET = {
    'tworoom': 'tworoom.h5',
    'pusht':   'pusht_expert_train.h5',
    'cube':    'ogbench/cube_single_expert.h5',
    'reacher': 'dmc/reacher_random.h5',
}[ENV]
ENV_ACTION_DIM = {'tworoom': 2, 'pusht': 2, 'cube': 5, 'reacher': 2}[ENV]

import os
# Ensure we're in the repo root
if not os.path.exists('train_idm.py'):
    for candidate in ['gc-idm', 'gc-idm-lite']:
        if os.path.exists(os.path.join(candidate, 'train_idm.py')):
            os.chdir(candidate)
            break
assert os.path.exists('train_idm.py'), 'cd to the repo root first'

os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['STABLEWM_HOME'] = os.path.abspath('./data')
os.environ['HF_HOME'] = os.path.abspath('./data/.hf_cache')
for k in ('STABLEWM_HOME', 'HF_HOME'):
    os.makedirs(os.environ[k], exist_ok=True)

print(f'ENV={ENV}  action_dim={ENV_ACTION_DIM}')
print(f'cwd={os.getcwd()}')
print(f'STABLEWM_HOME={os.environ["STABLEWM_HOME"]}')


In [ ]:
!apt-get update -qq && apt-get install -y -qq \
    libegl1 libgl1 libglew2.2 libosmesa6-dev \
    patchelf xvfb ffmpeg \
    libsdl2-dev libsdl2-image-dev zstd \
    > /dev/null 2>&1 && echo 'system deps OK'


## Download data

In [ ]:
# Download checkpoint (HuggingFace) + dataset (public Google Drive).
import os, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

HF_CKPT = {
    'tworoom': 'quentinll/lewm-tworooms',
    'pusht':   'quentinll/lewm-pusht',
    'cube':    'quentinll/lewm-cube',
    'reacher': 'quentinll/lewm-reacher',
}

# Public Google Drive file IDs for the four H5 datasets we use.
DRIVE_FILE_IDS = {
    'tworoom': '1_mgSY3PTUngCXe441J7bV-ddmUIWdGfH',
    'pusht':   '1JwsNso_ZWBh2FyVKObR4Km2XmjCN1-HM',
    'cube':    '10FnqNNVbLYoYdKS5kDP_L22-olSkeRuO',
    'reacher': '1Me0Eate7Z2j7Hjj4CKxKHalYWn2sUuze',
}

TARGET_REL = {
    'tworoom': 'tworoom.h5',
    'pusht':   'pusht_expert_train.h5',
    'cube':    'ogbench/cube_single_expert.h5',
    'reacher': 'dmc/reacher_random.h5',
}

stablewm = Path(os.environ['STABLEWM_HOME'])
ckpt_dir = stablewm / 'checkpoints' / ENV / 'lewm'
ds_dir   = stablewm / 'datasets'
ds_dir.mkdir(parents=True, exist_ok=True)
ckpt_dir.mkdir(parents=True, exist_ok=True)
target = ds_dir / TARGET_REL[ENV]

# (1) Checkpoint
if (ckpt_dir / 'config.json').exists() and (ckpt_dir / 'weights.pt').exists():
    print(f'  checkpoint already at {ckpt_dir}')
else:
    for f in ['config.json', 'weights.pt']:
        p = hf_hub_download(repo_id=HF_CKPT[ENV], filename=f, repo_type='model')
        shutil.copy(p, ckpt_dir / f)
    print(f'  checkpoint → {ckpt_dir}')

# (2) Dataset from public Google Drive (no zstd archive; raw .h5)
if target.exists():
    print(f'  dataset at {target} ({target.stat().st_size/1e9:.1f} GB)')
else:
    target.parent.mkdir(parents=True, exist_ok=True)
    file_id = DRIVE_FILE_IDS[ENV]
    url = f'https://drive.google.com/uc?id={file_id}'
    !gdown --fuzzy -O {target} '{url}'
    assert target.exists(), f'gdown failed to produce {target}'
    print(f'  dataset → {target} ({target.stat().st_size/1e9:.1f} GB)')


In [ ]:
# Verify
from pathlib import Path
home = Path(os.environ['STABLEWM_HOME'])
for rel in [f'datasets/{ENV_DATASET}', f'checkpoints/{ENV}/lewm/config.json', f'checkpoints/{ENV}/lewm/weights.pt']:
    p = home / rel
    sym = '✓' if p.exists() else '✗'
    sz = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'MISSING'
    print(f'  {sym} {rel:50s} {sz}')


## Extract embeddings


In [ ]:
import os
stablewm = os.environ['STABLEWM_HOME']
emb = f'{stablewm}/{ENV}_embeddings.npz'
if os.path.exists(emb):
    import numpy as np; d = np.load(emb)
    print(f'Already exists: {d["embeddings"].shape}')
else:
    !python train_idm.py extract \
        --checkpoint {stablewm}/checkpoints/{ENV}/lewm \
        --h5 {stablewm}/datasets/{ENV_DATASET} \
        --output {emb}


## Train GC-IDM


In [ ]:
import os
stablewm = os.environ['STABLEWM_HOME']
pt = f'{stablewm}/{ENV}_gcidm.pt'
!python train_idm.py train \
    --embeddings {stablewm}/{ENV}_embeddings.npz \
    --output {pt} \
    --embed-dim 192 --action-dim {ENV_ACTION_DIM} --frameskip 1 \
    --hidden-dim 512 --n-layers 3 \
    --noise-sigma 0.0 --max-goal-horizon 50 \
    --lr 1e-3 --batch-size 1024 --epochs 50 --seed 42

## Evaluate GC-IDM vs CEM


In [ ]:
import os
stablewm = os.environ['STABLEWM_HOME']
pt = f'{stablewm}/{ENV}_gcidm.pt'
!python eval_idm.py \
    --dataset {ENV} --idm {pt} \
    --compare \
    --num-eval 200 --eval-budget 50 --goal-offset 25 --seed 42

## Other solvers (MPPI, iCEM, Gradient)


In [ ]:
for solver in ['mppi', 'icem', 'gradient']:
    print(f'\n{"="*60}\n  {solver.upper()}\n{"="*60}')
    !python eval_othersolvers.py --solver {solver} --dataset {ENV} --num-eval 200 --eval-budget 50 --goal-offset 25 --seed 42